# Results

In [1]:
import torch
import timm
import tome
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from eval import evaluate
from SReT import SReT_T_distill
import SReT_ToMe

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")

batch_size = 128
dataset_dir = '/media/datasets/imagenet/val'
dataset_transform = transforms.Compose([
    transforms.Resize(256), 
    transforms.CenterCrop(224), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(dataset_dir, transform=dataset_transform)
dataset_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
print(f'Loaded {len(dataset)} ImageNet-1k Validation images')

CUDA available: True
GPU Device Name: NVIDIA GeForce RTX 4060 Ti
Current GPU Device: 0
Loaded 50000 ImageNet-1k Validation images


## Evaluate

#### DeiT-Tiny-Distill + ToMe

In [4]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()

rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model.r = r
    _ = evaluate(model, dataset_loader, batch_size)

--- r = 0 ---
 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         74.40 %
 Total Parameters:       5.91 M
 Theoretical FLOPs:      2.17 G
 Throughput (BS=128):    1769.22 images/sec
 Throughput (BS=64):    1891.94 images/sec
 Throughput (BS=32):    1960.87 images/sec
 Throughput (BS=16):    2374.62 images/sec
 Throughput (BS=1):    648.88 images/sec
 Model Weights VRAM:     106.18 MB
 Peak Activation Memory: 247.01 MB

--- r = 10 ---
 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         73.47 %
 Total Parameters:       5.91 M
 Theoretical FLOPs:      1.51 G
 Throughput (BS=128):    2405.74 images/sec
 Throughput (BS=64):    2624.78 images/sec
 Throughput (BS=32):    2669.12 images/sec
 Throughput (BS=16):    2634.93 images/sec
 Throughput (BS=1):    237.51 images/sec
 Model Weights VRAM:     106.22 MB
 Peak Activation Memory: 232.62 MB

--- r = 15 ---
 Target Batch Size:      128
---

#### PiT

In [2]:
# tiny
model = timm.create_model("pit_ti_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         74.42 %
 Total Parameters:       5.10 M
 Theoretical FLOPs:      1.01 G
 Throughput (BS=128):    1794.22 images/sec
 Throughput (BS=64):    1851.28 images/sec
 Throughput (BS=32):    1956.55 images/sec
 Throughput (BS=16):    2064.07 images/sec
 Throughput (BS=1):    661.18 images/sec
 Model Weights VRAM:     103.09 MB
 Peak Activation Memory: 1203.36 MB



In [4]:
# xs
model = timm.create_model("pit_xs_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         79.13 %
 Total Parameters:       11.00 M
 Theoretical FLOPs:      2.21 G
 Throughput (BS=128):    1354.16 images/sec
 Throughput (BS=64):    1402.87 images/sec
 Throughput (BS=32):    1443.19 images/sec
 Throughput (BS=16):    1538.35 images/sec
 Throughput (BS=1):    668.56 images/sec
 Model Weights VRAM:     125.18 MB
 Peak Activation Memory: 1283.77 MB



#### EfficientViT-M2

In [2]:
efficientvit_model = timm.create_model('efficientvit_m2.r224_in1k', pretrained=True)
efficientvit_model = efficientvit_model.cuda().eval()
_ = evaluate(efficientvit_model, dataset_loader, batch_size)

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         70.61 %
 Total Parameters:       4.19 M
 Theoretical FLOPs:      0.41 G
 Throughput (BS=128):    7648.81 images/sec
 Throughput (BS=64):    8182.80 images/sec
 Throughput (BS=32):    7808.96 images/sec
 Throughput (BS=16):    4695.23 images/sec
 Throughput (BS=1):    301.64 images/sec
 Model Weights VRAM:     99.03 MB
 Peak Activation Memory: 195.51 MB



#### SReT-Tiny-Distill

In [2]:
model = SReT_T_distill(pretrained=False) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         77.36 %
 Total Parameters:       4.76 M
 Theoretical FLOPs:      1.91 G
 Throughput (BS=128):    1068.60 images/sec
 Throughput (BS=64):    1169.38 images/sec
 Throughput (BS=32):    1250.31 images/sec
 Throughput (BS=16):    1299.71 images/sec
 Throughput (BS=1):    221.74 images/sec
 Model Weights VRAM:     92.20 MB
 Peak Activation Memory: 1274.10 MB



#### SReT + ToMe
- Constant token reduction

In [3]:
rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model = SReT_ToMe.SReT_T_distill(pretrained=False, constant_r=r) # initialize
    checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
    model.load_state_dict(checkpoint['model'])
    model = model.cuda().eval()
    _ = evaluate(model, dataset_loader, batch_size)

--- r = 0 ---
 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         77.39 %
 Total Parameters:       4.76 M
 Theoretical FLOPs:      1.91 G
 Throughput (BS=128):    964.64 images/sec
 Throughput (BS=64):    1045.45 images/sec
 Throughput (BS=32):    1126.49 images/sec
 Throughput (BS=16):    1191.04 images/sec
 Throughput (BS=1):    196.49 images/sec
 Model Weights VRAM:     100.82 MB
 Peak Activation Memory: 785.15 MB

--- r = 10 ---
 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         70.83 %
 Total Parameters:       4.76 M
 Theoretical FLOPs:      1.32 G
 Throughput (BS=128):    1173.56 images/sec
 Throughput (BS=64):    1270.38 images/sec
 Throughput (BS=32):    1315.02 images/sec
 Throughput (BS=16):    1151.86 images/sec
 Throughput (BS=1):    108.11 images/sec
 Model Weights VRAM:     100.82 MB
 Peak Activation Memory: 770.99 MB

--- r = 15 ---
 Target Batch Size:      128
----

#### SReT + ToMe
- Dynamic Decaying Schedule
- Best result from grid search results (r=0.3, alpha=0.1)

In [3]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, initial_r_ratio=0.3, alpha=0.1) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         74.99 %
 Total Parameters:       4.76 M
 Theoretical FLOPs:      1.37 G
 Throughput (BS=128):    1507.24 images/sec
 Throughput (BS=64):    1673.48 images/sec
 Throughput (BS=32):    1741.15 images/sec
 Throughput (BS=16):    1760.40 images/sec
 Throughput (BS=1):    153.58 images/sec
 Model Weights VRAM:     101.81 MB
 Peak Activation Memory: 437.18 MB

